# Notebook 10 — RNA-seq: Spliced Alignment and STAR

**Module:** 09 — Genomics and NGS  
**Notebook:** 10 of 16  
**Time estimate:** 75 minutes

> RNA-seq alignment is fundamentally different from DNA alignment because reads
> can span exon-exon junctions. Know STAR's two-pass strategy and the CIGAR 'N' op.

---
## Step 1 — Motivation

mRNA has been processed: introns removed, exons joined. An RNA-seq read from an
exon-exon junction is split across a genomic gap that can be hundreds of kilobases.
BWA-MEM cannot handle this — it would soft-clip the read at the junction. STAR
finds the junction and reports a split alignment with a CIGAR 'N' operation.

---
## Step 2 — Intuition

**STAR's maximal mappable prefix (MMP) approach:**

1. Find the longest prefix of the read that maps exactly to the genome (MMP)
2. If the MMP doesn't cover the full read, mark the end of the MMP as a potential
   donor splice site (GT...)
3. Separately map the suffix of the read from the other end
4. Find the genomic gap between MMP end and suffix start that best fits splice
   site motifs (GT...AG canonical, GC...AG, AT...AC non-canonical)
5. Report the split alignment: `prefix_len + M + intron_len + N + suffix_len + M`

**Two-pass alignment:**
- **Pass 1:** Align all reads, discover novel splice junctions (not in the annotation)
- **Pass 2:** Re-align all reads using the annotation + novel junctions from Pass 1
- Result: significantly better alignment of reads spanning novel junctions
  (important for alternative splicing analysis)

---
## Step 3 — Biological Background

**Splice junction motifs:**

| Junction type | Donor | Acceptor | Frequency |
|--------------|-------|----------|----------|
| Canonical U2 | GT | AG | ~98.7% |
| Canonical U12 (minor) | AT | AC | ~0.04% |
| GC-AG | GC | AG | ~0.7% |

The GT...AG rule means introns almost always start with GT and end with AG.
STAR uses this to score candidate splice junctions.

**Why RNA-seq coverage is not uniform:**
1. **3' bias:** Poly-A capture libraries are enriched at 3' ends of transcripts
2. **GC content bias:** Same as WGS — PCR efficiency varies with GC
3. **Transcript abundance:** More abundant transcripts have more reads
4. **Read-through transcription:** Reads past the poly-A site

**STAR output (SJ.out.tab):**  
Per-junction file showing: chrom, start, end, strand, intron motif,
annotated (yes/no), number of reads crossing junction, max overhang

---
## Step 4 — Mathematical Explanation

**CIGAR 'N' operation for spliced reads:**

A read spanning a 5000 bp intron with 50 bp in exon 1 and 100 bp in exon 2:
```
CIGAR: 50M5000N100M
POS:   10000
→ aligns to chr1:10000-10049 (exon 1) and chr1:15050-15149 (exon 2)
Reference consumed: 50 + 5000 + 100 = 5150 bp
Query consumed: 50 + 100 = 150 bp
```

**Splice junction score (STAR):**
$$S_{junc} = (\text{junction motif score}) + \sum_{r} \text{overhang}_r$$

where overhang is the number of bases of the read within the exon at the junction.
Minimum overhang threshold (default 8 bp) prevents false junctions from poorly
aligned reads.

**Unique vs. multi-mapping in RNA-seq:**
- Multi-mapping is more common in RNA-seq than WGS due to paralogs and
  repetitive elements in transcribed regions
- STAR `--outFilterMultimapNmax 1` keeps only unique mappers
- featureCounts has `--fraction` mode to split multi-mapper counts

In [ ]:
# Step 6 — Python: Splice junction detection and spliced read simulation
import numpy as np
from dataclasses import dataclass


@dataclass
class Gene:
    name: str
    chrom: str
    strand: str
    exons: list[tuple[int, int]]  # (start, end) 0-based

    @property
    def transcript_length(self) -> int:
        return sum(end - start for start, end in self.exons)

    def get_transcript_sequence(self, genome: str) -> str:
        seq = ''.join(genome[s:e] for s, e in self.exons)
        if self.strand == '-':
            comp = {'A':'T','T':'A','G':'C','C':'G','N':'N'}
            seq = ''.join(comp.get(b,'N') for b in reversed(seq))
        return seq

    def introns(self) -> list[tuple[int, int]]:
        result = []
        for i in range(len(self.exons) - 1):
            result.append((self.exons[i][1], self.exons[i+1][0]))
        return result


def simulate_genome(length: int = 50000, seed: int = 42) -> str:
    rng = np.random.default_rng(seed)
    return ''.join(rng.choice(['A','C','G','T'], length))


def plant_splice_sites(genome: list, introns: list[tuple[int,int]]):
    """Plant GT...AG splice sites at intron boundaries in genome."""
    genome = list(genome)
    for start, end in introns:
        if start + 2 < len(genome):
            genome[start] = 'G'
            genome[start+1] = 'T'
        if end - 2 >= 0:
            genome[end-2] = 'A'
            genome[end-1] = 'G'
    return ''.join(genome)


def simulate_rnaseq_reads(
    gene: Gene,
    genome: str,
    n_reads: int = 100,
    read_length: int = 75,
    error_rate: float = 0.005,
    seed: int = 42
) -> list[dict]:
    """
    Simulate RNA-seq reads from a gene's transcript.
    Returns list of {sequence, true_cigar, true_pos, junction_spanning}.
    """
    rng = np.random.default_rng(seed)
    transcript = gene.get_transcript_sequence(genome)
    tlen = len(transcript)
    if tlen < read_length:
        return []

    # Build transcript → genome coordinate map
    t_to_g = []
    for start, end in gene.exons:
        t_to_g.extend(range(start, end))

    reads = []
    for _ in range(n_reads):
        t_start = int(rng.integers(0, tlen - read_length + 1))
        t_end = t_start + read_length
        read_seq = list(transcript[t_start:t_end])

        # Add errors
        for j in range(len(read_seq)):
            if rng.random() < error_rate:
                read_seq[j] = rng.choice([b for b in 'ACGT' if b != read_seq[j]])

        # Build CIGAR
        g_coords = t_to_g[t_start:t_end]
        cigar_ops = []
        i = 0
        junction_spans = False
        while i < len(g_coords):
            j = i + 1
            while j < len(g_coords) and g_coords[j] == g_coords[j-1] + 1:
                j += 1
            cigar_ops.append(f'{j-i}M')
            if j < len(g_coords):
                intron_len = g_coords[j] - g_coords[j-1] - 1
                cigar_ops.append(f'{intron_len}N')
                junction_spans = True
            i = j

        reads.append({
            'sequence': ''.join(read_seq),
            'cigar': ''.join(cigar_ops),
            'pos': g_coords[0] + 1,  # 1-based SAM position
            'junction_spanning': junction_spans,
        })
    return reads


# Simulate a 3-exon gene
genome = simulate_genome(50000)
gene = Gene(
    name='GENE1',
    chrom='chr1',
    strand='+',
    exons=[(1000, 1200), (3000, 3300), (6000, 6400)]
)
# Plant splice sites
genome = plant_splice_sites(genome, gene.introns())

reads = simulate_rnaseq_reads(gene, genome, n_reads=200, read_length=75)
junction_reads = [r for r in reads if r['junction_spanning']]
exonic_reads = [r for r in reads if not r['junction_spanning']]

print(f"Gene: {gene.name}, transcript length: {gene.transcript_length} bp")
print(f"Introns: {gene.introns()}")
print(f"Total reads: {len(reads)}")
print(f"Junction-spanning reads: {len(junction_reads)} ({100*len(junction_reads)/len(reads):.1f}%)")
print(f"Exon-only reads:        {len(exonic_reads)}")
print()
print("Sample junction-spanning reads:")
for r in junction_reads[:3]:
    print(f"  pos={r['pos']} cigar={r['cigar']} seq={r['sequence'][:20]}...")

In [ ]:
# Detect splice junctions from simulated reads
def detect_splice_junctions(reads: list[dict]) -> list[dict]:
    """Extract splice junctions from CIGAR N operations."""
    import re
    junctions = {}
    for r in reads:
        if 'N' not in r['cigar']:
            continue
        pos = r['pos'] - 1  # 0-based
        ops = re.findall(r'(\d+)([MIDNSHP=X])', r['cigar'])
        for length, op in ops:
            length = int(length)
            if op == 'M':
                pos += length
            elif op == 'N':
                junc_key = (pos, pos + length)
                if junc_key not in junctions:
                    junctions[junc_key] = {'start': pos, 'end': pos + length, 'count': 0}
                junctions[junc_key]['count'] += 1
                pos += length
            elif op == 'D':
                pos += length
    return sorted(junctions.values(), key=lambda j: j['count'], reverse=True)


junctions = detect_splice_junctions(reads)
print("Detected splice junctions:")
print(f"{'Start':>8} {'End':>8} {'Length':>8} {'Read count':>12}")
print('-' * 42)
for j in junctions:
    print(f"{j['start']:>8} {j['end']:>8} {j['end']-j['start']:>8} {j['count']:>12}")

print(f"\nTrue intron positions: {[(s, e) for s, e in gene.introns()]}")

In [ ]:
# Step 7 — Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Gene structure and read coverage
ax = axes[0, 0]
# Coverage vector
cov_len = 8000
cov = np.zeros(cov_len)
import re
for r in reads:
    pos = r['pos'] - 1
    for length, op in re.findall(r'(\d+)([MIDNSHP=X])', r['cigar']):
        length = int(length)
        if op == 'M' and pos < cov_len and pos + length <= cov_len:
            cov[pos:pos+length] += 1
        if op in 'MDN': pos += length

ax.fill_between(range(cov_len), cov, alpha=0.7, color='steelblue')
# Draw exon blocks
for i, (s, e) in enumerate(gene.exons):
    ax.add_patch(plt.Rectangle((s, -5), e-s, 4, facecolor='orange', edgecolor='darkorange', lw=1.5))
    ax.text((s+e)/2, -3, f'Exon {i+1}', ha='center', fontsize=8)
# Draw introns
for s, e in gene.introns():
    ax.plot([s, e], [-3, -3], 'k-', lw=1)
ax.set_xlim(800, 7000)
ax.set_ylim(-8, cov.max() + 5)
ax.set_xlabel('Genomic position')
ax.set_ylabel('Coverage')
ax.set_title('A. RNA-seq coverage over 3-exon gene\n(coverage drops to 0 in introns)')

# Panel B: CIGAR N illustration
ax2 = axes[0, 1]
ax2.axis('off')
# Show two read alignments
examples = [
    ('75M', 'Exon-only read (no junction)'),
    ('50M1800N25M', 'Junction-spanning (exon1-exon2)'),
    ('30M3700N45M', 'Junction-spanning (exon2-exon3)'),
]
for i, (cigar, desc) in enumerate(examples):
    y = 0.85 - i * 0.28
    ax2.text(0.05, y + 0.05, f'CIGAR: {cigar}', fontsize=11, fontfamily='monospace',
             fontweight='bold')
    ax2.text(0.05, y, desc, fontsize=9, color='gray')
    # Visual bar
    ops = re.findall(r'(\d+)([MIDNSHP=X])', cigar)
    total = sum(int(l) for l, op in ops if op in 'MN')
    x = 0.05
    for length_s, op in ops:
        length = int(length_s)
        w = 0.7 * length / total
        color = 'steelblue' if op == 'M' else 'lightgray'
        ax2.add_patch(plt.Rectangle((x, y - 0.10), w, 0.07,
                                      facecolor=color, edgecolor='gray', lw=0.5))
        if op == 'M':
            ax2.text(x + w/2, y - 0.065, 'M', ha='center', fontsize=7, color='white')
        elif op == 'N':
            ax2.text(x + w/2, y - 0.065, 'N (intron)', ha='center', fontsize=7, color='black')
        x += w
ax2.set_title('B. CIGAR strings for RNA-seq reads\n(N = intron skip)', fontweight='bold', fontsize=10)
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)

# Panel C: Junction-spanning read fraction by read length
ax3 = axes[1, 0]
read_lengths = [50, 75, 100, 125, 150]
junc_fracs = []
for rl in read_lengths:
    r = simulate_rnaseq_reads(gene, genome, n_reads=500, read_length=rl, seed=1)
    jf = sum(1 for x in r if x['junction_spanning']) / len(r)
    junc_fracs.append(jf * 100)
ax3.plot(read_lengths, junc_fracs, 'bo-', lw=2, ms=7)
ax3.set_xlabel('Read length (bp)')
ax3.set_ylabel('% junction-spanning reads')
ax3.set_title('C. Junction-spanning read fraction\nvs. read length')

# Panel D: DNA vs RNA alignment comparison table
ax4 = axes[1, 1]
ax4.axis('off')
table_data = [
    ['Feature', 'DNA (BWA-MEM)', 'RNA (STAR)'],
    ['Input', 'Genomic DNA reads', 'mRNA-derived reads'],
    ['Gap type allowed', 'Small indels only', 'Introns (>1 kb)'],
    ['Junction-aware', 'No', 'Yes (GT-AG rule)'],
    ['CIGAR ops used', 'M,I,D,S', 'M,I,D,S,N'],
    ['Typical speed', 'Very fast', 'Moderate (genome index)'],
    ['Two-pass mode', 'N/A', 'Yes — discovers novel junctions'],
]
t = ax4.table(table_data, loc='center', cellLoc='left')
t.auto_set_font_size(False); t.set_fontsize(8)
t.scale(1, 1.5)
# Header row styling
for j in range(3):
    t[0,j].set_facecolor('#D0E8FF')
    t[0,j].set_text_props(fontweight='bold')
ax4.set_title('D. DNA vs. RNA alignment comparison', fontweight='bold', fontsize=10)

plt.suptitle('RNA-seq Spliced Alignment: STAR Concepts', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rnaseq_spliced_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. A read has CIGAR `35M2145N40M`. It starts at genomic position 10000. What
   genomic positions does it cover? What are the exon boundaries?
2. Implement `check_splice_site(genome, intron_start, intron_end)` that checks
   if a detected junction has a canonical GT...AG motif.
3. Why does RNA-seq coverage drop to zero in introns even at high sequencing depth?
   (This seems obvious — but state it precisely in terms of what the library contains.)

---
## Step 10 — Quiz

1. What does the CIGAR 'N' operation mean in RNA-seq alignments?
2. What is STAR's two-pass alignment strategy and why does it improve results?
3. What splice site motif does STAR use to validate junctions?
4. Why can BWA-MEM not align RNA-seq reads that span exon-exon junctions?
5. What is an SJ.out.tab file produced by STAR?

---
## Step 12 — Reflection

> *[In one paragraph: explain why a junction-spanning read is evidence for a specific
> exon-exon junction. What would it look like in a SAM file, and what does each
> part of the CIGAR string tell you?]*

---
*Next: `11_transcript_quantification_tpm_counts.ipynb`*